# Ensemble BO for f4–f6

Updated strategy:
- ARD GP
- RandomForest importance
- LASSO screening as soft weighting
- Dimension-scaled distance in candidate generation / diversity bonus
- **Optional local neural-network helper** for nonlinear ranking only
- Backpropagation used only for:
  - directional candidate proposals
  - coordinate reweighting
  - local refinement around incumbent best points
- EI / UCB remain the acquisition decision rule


In [1]:
import numpy as np
import pandas as pd
import math
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LassoCV

import torch
import torch.nn as nn

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# --- Config ---
DATA_PATH = Path("data.csv")
FUNCTIONS = ['function_4', 'function_5', 'function_6']
DIMS = {'function_4': 4, 'function_5': 4, 'function_6': 5}
CURRENT_STEP = 9
N_GLOBAL = 3000
N_LOCAL = 1400
LOW, HIGH = 0.0, 1.0
RANDOM_SEED = 19
IS_MAXIMISATION = True
DEBUG = True
DISTANCE_BONUS = 0.06
LOCAL_RADIUS = 0.12
USE_NN_LOCAL_HELPER = True
NN_MIN_POINTS = 14
NN_MAX_EPOCHS = 120
NN_RANK_BONUS = 0.08
GRAD_WEIGHT_BLEND = 0.18
GRAD_CANDIDATES = 24

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.set_num_threads(1)
torch.set_num_interop_threads(1)


def normal_pdf(z):
    return (1.0 / np.sqrt(2.0 * np.pi)) * np.exp(-0.5 * z**2)


def normal_cdf(z):
    return 0.5 * (1.0 + np.vectorize(math.erf)(z / np.sqrt(2.0)))


def acquisition_ucb(mu, sigma, kappa):
    return mu + kappa * sigma


def acquisition_ei(mu, sigma, best_y, xi):
    sigma = np.maximum(sigma, 1e-12)
    imp = mu - best_y - xi
    z = imp / sigma
    return imp * normal_cdf(z) + sigma * normal_pdf(z)


def read_data(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Could not find {path}.")
    df = pd.read_csv(path)
    df["function"] = df["function"].astype(str).str.strip()
    return df


def get_xy(df: pd.DataFrame, fname: str, d: int):
    sub = df[df["function"] == fname].copy()
    if sub.empty:
        raise ValueError(f"No rows found for function={fname} in {DATA_PATH}.")
    X = sub[[f"x{i}" for i in range(1, d + 1)]].to_numpy(dtype=float)
    y = sub["y"].to_numpy(dtype=float).reshape(-1)
    m = np.isfinite(X).all(axis=1) & np.isfinite(y)
    return X[m], y[m]


def format_output_line(fname, x):
    parts = [fname] + [f"{v:.6f}" for v in x.tolist()]
    return ", ".join(parts)


def minmax01(a):
    a = np.asarray(a, dtype=float)
    lo, hi = np.min(a), np.max(a)
    if hi - lo < 1e-12:
        return np.zeros_like(a)
    return (a - lo) / (hi - lo)


def safe_normalise(w, floor=1e-3):
    w = np.asarray(w, dtype=float)
    w = np.where(np.isfinite(w), w, 0.0)
    w = np.maximum(w, 0.0)
    if w.sum() <= 0:
        w = np.ones_like(w)
    w = w / w.sum()
    w = np.maximum(w, floor)
    return w / w.sum()


def weights_to_scales(w, clip=(0.35, 2.5)):
    w = safe_normalise(w)
    scales = np.sqrt(w * w.size)
    return np.clip(scales, clip[0], clip[1])


def scaled_distance(X, x0, w):
    X = np.asarray(X, dtype=float)
    x0 = np.asarray(x0, dtype=float)
    w = safe_normalise(w)
    diff = X - x0.reshape(1, -1)
    return np.sqrt(np.sum(w.reshape(1, -1) * diff * diff, axis=1))


def min_scaled_distance_to_observed(C, X, w):
    w = safe_normalise(w)
    diff = C[:, None, :] - X[None, :, :]
    d2 = np.sum(w.reshape(1, 1, -1) * diff * diff, axis=2)
    return np.sqrt(np.min(d2, axis=1))


def sample_uniform(n, d, rng):
    return rng.uniform(LOW, HIGH, size=(n, d))


def sample_local_weighted(best_x, n, rng, radius, dim_weights):
    scales = weights_to_scales(dim_weights)
    X = best_x.reshape(1, -1) + rng.normal(0.0, radius, size=(n, best_x.size)) * scales.reshape(1, -1)
    return np.clip(X, LOW, HIGH)


def schedule_params(step_idx):
    if step_idx <= 5:
        return "ucb", 4.5, None
    if step_idx <= 9:
        return "ucb", 2.5, 0.03
    return "ei", None, 0.01


def fit_gp(X, y, seed=0):
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    d = X.shape[1]
    kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(
        length_scale=np.ones(d), length_scale_bounds=(1e-3, 1e3), nu=2.5
    ) + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-1))
    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=2,
        random_state=seed,
    )
    gp.fit(Xs, y)
    return gp, scaler


def extract_length_scales(kernel):
    if hasattr(kernel, "k1") and hasattr(kernel.k1, "k2") and hasattr(kernel.k1.k2, "length_scale"):
        return np.asarray(kernel.k1.k2.length_scale, dtype=float).reshape(-1)
    if hasattr(kernel, "length_scale"):
        return np.asarray(kernel.length_scale, dtype=float).reshape(-1)
    for attr in ["k1", "k2"]:
        if hasattr(kernel, attr):
            out = extract_length_scales(getattr(kernel, attr))
            if out is not None:
                return out
    return None


def ard_weights_from_gp(gp, d):
    ls = extract_length_scales(gp.kernel_)
    if ls is None:
        return np.ones(d) / d
    if ls.size == 1:
        ls = np.repeat(ls, d)
    rel = 1.0 / np.maximum(ls, 1e-9)
    return safe_normalise(rel)


def fit_rf(X, y, seed=0):
    rf = RandomForestRegressor(
        n_estimators=160,
        max_depth=None,
        min_samples_leaf=2,
        random_state=seed,
    )
    rf.fit(X, y)
    return rf


def rf_predict_mean_std(rf, Xcand):
    preds = np.column_stack([t.predict(Xcand) for t in rf.estimators_])
    return preds.mean(axis=1), np.maximum(preds.std(axis=1, ddof=0), 1e-12)


def fit_lasso_screen(X, y, seed=0):
    scaler_x = StandardScaler()
    scaler_y = StandardScaler()
    Xs = scaler_x.fit_transform(X)
    ys = scaler_y.fit_transform(y.reshape(-1, 1)).ravel()
    cv = max(2, min(5, len(y) - 1))
    model = LassoCV(cv=cv, random_state=seed, max_iter=20000)
    model.fit(Xs, ys)
    coef = np.abs(model.coef_)
    if coef.sum() <= 0:
        coef = np.ones(X.shape[1])
    return safe_normalise(coef)


class LocalMLP(nn.Module):
    def __init__(self, d, hidden=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, hidden),
            nn.Tanh(),
            nn.Linear(hidden, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x)


def select_local_subset(X, y, center, dim_weights, max_points=None):
    n = len(X)
    if max_points is None:
        max_points = min(n, max(12, 3 * X.shape[1] + 6))
    d = scaled_distance(X, center, dim_weights)
    idx = np.argsort(d)[:max_points]
    return X[idx], y[idx], idx


def fit_local_mlp(X, y, seed=0):
    torch.manual_seed(seed)
    x_mean = X.mean(axis=0)
    x_std = X.std(axis=0)
    x_std = np.where(x_std < 1e-8, 1.0, x_std)
    y_mean = float(np.mean(y))
    y_std = float(np.std(y))
    y_std = 1.0 if y_std < 1e-8 else y_std

    Xs = ((X - x_mean) / x_std).astype(np.float32)
    ys = ((y - y_mean) / y_std).astype(np.float32).reshape(-1, 1)

    model = LocalMLP(X.shape[1], hidden=min(24, max(12, 4 * X.shape[1])))
    opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)
    loss_fn = nn.MSELoss()
    Xt = torch.tensor(Xs)
    yt = torch.tensor(ys)

    model.train()
    for epoch in range(NN_MAX_EPOCHS):
        opt.zero_grad()
        pred = model(Xt)
        loss = loss_fn(pred, yt)
        loss.backward()
        opt.step()
    payload = {
        "model": model.eval(),
        "x_mean": torch.tensor(x_mean.astype(np.float32)),
        "x_std": torch.tensor(x_std.astype(np.float32)),
        "y_mean": y_mean,
        "y_std": y_std,
    }
    return payload


def mlp_predict(payload, Xcand):
    model = payload["model"]
    x_mean = payload["x_mean"]
    x_std = payload["x_std"]
    Xt = torch.tensor(Xcand.astype(np.float32))
    Xs = (Xt - x_mean) / x_std
    with torch.no_grad():
        pred = model(Xs).squeeze(-1).cpu().numpy()
    return payload["y_mean"] + payload["y_std"] * pred


def mlp_input_gradient(payload, x_raw):
    model = payload["model"]
    x_mean = payload["x_mean"]
    x_std = payload["x_std"]
    x = torch.tensor(x_raw.astype(np.float32), requires_grad=True)
    xs = (x - x_mean) / x_std
    yhat = payload["y_mean"] + payload["y_std"] * model(xs).squeeze()
    yhat.backward()
    grad = x.grad.detach().cpu().numpy()
    return grad


def gradient_weights_from_vec(grad):
    return safe_normalise(np.abs(np.asarray(grad, dtype=float)))


def gradient_direction_candidates(best_x, grad_vec, dim_weights, radius, n_steps=6):
    grad_vec = np.asarray(grad_vec, dtype=float)
    if not np.isfinite(grad_vec).all() or np.linalg.norm(grad_vec) < 1e-12:
        return np.empty((0, best_x.size))
    direction = grad_vec / (np.linalg.norm(grad_vec) + 1e-12)
    scales = weights_to_scales(dim_weights, clip=(0.25, 2.0))
    pts = []
    for alpha in np.linspace(0.25, 1.0, n_steps):
        step = alpha * radius * direction * scales
        pts.append(np.clip(best_x + step, LOW, HIGH))
        pts.append(np.clip(best_x - 0.35 * step, LOW, HIGH))
    return np.vstack(pts)


def refinement_candidates(best_x, grad_vec, dim_weights, rng, radius, n=12):
    if not np.isfinite(grad_vec).all() or np.linalg.norm(grad_vec) < 1e-12:
        return np.empty((0, best_x.size))
    g = grad_vec / (np.linalg.norm(grad_vec) + 1e-12)
    gw = gradient_weights_from_vec(grad_vec)
    combo = safe_normalise(0.6 * safe_normalise(dim_weights) + 0.4 * gw)
    scales = weights_to_scales(combo, clip=(0.25, 2.0))
    pts = []
    for _ in range(n):
        jitter = rng.normal(0.0, 0.35 * radius, size=best_x.size) * scales
        drift = 0.4 * radius * g * scales
        pts.append(np.clip(best_x + drift + jitter, LOW, HIGH))
    return np.vstack(pts)


def ensemble_predict(gp, scaler_gp, rf, Xcand):
    Xs = scaler_gp.transform(Xcand)
    mu_gp, std_gp = gp.predict(Xs, return_std=True)
    mu_rf, std_rf = rf_predict_mean_std(rf, Xcand)
    mu_ens = 0.65 * mu_gp + 0.35 * mu_rf
    var_ens = (0.65 ** 2) * (std_gp ** 2) + (0.35 ** 2) * (std_rf ** 2) + 0.15 * (mu_gp - mu_rf) ** 2
    return mu_ens, np.sqrt(np.maximum(var_ens, 1e-12))


def combined_dimension_weights(gp, rf, lasso_w):
    ard_w = ard_weights_from_gp(gp, len(rf.feature_importances_))
    rf_w = safe_normalise(rf.feature_importances_)
    combo = 0.55 * ard_w + 0.30 * rf_w + 0.15 * lasso_w
    return safe_normalise(combo), ard_w, rf_w


def propose_next(fname, X, y, rng):
    mode, kappa, xi = schedule_params(CURRENT_STEP)
    best_idx = int(np.argmax(y))
    best_x = X[best_idx].copy()
    best_y = float(y[best_idx])

    gp, scaler = fit_gp(X, y, seed=RANDOM_SEED)
    rf = fit_rf(X, y, seed=RANDOM_SEED)
    lasso_w = fit_lasso_screen(X, y, seed=RANDOM_SEED)
    dim_weights, ard_w, rf_w = combined_dimension_weights(gp, rf, lasso_w)

    mlp_payload = None
    grad_vec = None
    grad_w = None
    nn_local_points = 0

    if USE_NN_LOCAL_HELPER and len(X) >= NN_MIN_POINTS:
        X_loc, y_loc, _ = select_local_subset(X, y, best_x, dim_weights)
        nn_local_points = len(X_loc)
        if len(X_loc) >= NN_MIN_POINTS:
            try:
                mlp_payload = fit_local_mlp(X_loc, y_loc, seed=RANDOM_SEED)
                grad_vec = mlp_input_gradient(mlp_payload, best_x)
                grad_w = gradient_weights_from_vec(grad_vec)
                dim_weights = safe_normalise((1.0 - GRAD_WEIGHT_BLEND) * dim_weights + GRAD_WEIGHT_BLEND * grad_w)
            except Exception:
                mlp_payload = None
                grad_vec = None
                grad_w = None

    pools = [
        sample_uniform(N_GLOBAL, X.shape[1], rng),
        sample_local_weighted(best_x, N_LOCAL, rng, radius=LOCAL_RADIUS, dim_weights=dim_weights),
    ]
    if mlp_payload is not None and grad_vec is not None:
        pools.append(gradient_direction_candidates(best_x, grad_vec, dim_weights, radius=LOCAL_RADIUS, n_steps=max(4, GRAD_CANDIDATES // 4)))
        pools.append(refinement_candidates(best_x, grad_vec, dim_weights, rng, radius=0.75 * LOCAL_RADIUS, n=GRAD_CANDIDATES))
    C = np.vstack([p for p in pools if len(p) > 0])
    C = np.unique(np.round(C, 8), axis=0)

    mu, std = ensemble_predict(gp, scaler, rf, C)
    score = acquisition_ucb(mu, std, kappa=kappa) if mode == "ucb" else acquisition_ei(mu, std, best_y=best_y, xi=xi)
    score = score + DISTANCE_BONUS * minmax01(min_scaled_distance_to_observed(C, X, dim_weights))

    if mlp_payload is not None:
        nn_rank = minmax01(mlp_predict(mlp_payload, C))
        score = score + NN_RANK_BONUS * nn_rank

    if DEBUG:
        extra = ""
        if grad_w is not None:
            extra = f" GRAD={np.round(grad_w,3)} NN_local_n={nn_local_points}"
        print(f"[{fname}] ARD={np.round(ard_w,3)} RF={np.round(rf_w,3)} LASSO={np.round(lasso_w,3)} COMBO={np.round(dim_weights,3)}{extra}")
    return C[int(np.argmax(score))]


df = read_data(DATA_PATH)
rng = np.random.default_rng(RANDOM_SEED)
for fname in FUNCTIONS:
    d = DIMS[fname]
    X, y = get_xy(df, fname, d)
    x_next = propose_next(fname, X, y, rng)
    print(format_output_line(fname, x_next))


[function_4] ARD=[0.257 0.262 0.229 0.252] RF=[0.207 0.238 0.157 0.397] LASSO=[0.19  0.128 0.175 0.507] COMBO=[0.312 0.197 0.207 0.283] GRAD=[0.679 0.025 0.243 0.053] NN_local_n=18
function_4, 0.384719, 0.263201, 0.349899, 0.431782
[function_5] ARD=[0.42  0.251 0.199 0.131] RF=[0.006 0.521 0.349 0.123] LASSO=[0.11  0.304 0.397 0.188] COMBO=[0.23  0.307 0.295 0.167] GRAD=[0.143 0.159 0.394 0.303] NN_local_n=18
function_5, 0.685362, 0.907407, 1.000000, 1.000000
[function_6] ARD=[0.236 0.221 0.149 0.177 0.216] RF=[0.106 0.066 0.109 0.354 0.364] LASSO=[0.075 0.15  0.228 0.21  0.337] COMBO=[0.196 0.175 0.146 0.197 0.286] GRAD=[0.303 0.227 0.13  0.021 0.319] NN_local_n=21
function_6, 0.441590, 0.473055, 0.829400, 0.955744, 0.257643
